# RealData Graph Retrieval and NL-to-Cypher for PostgreSQL AGE

This notebook is the retrieval/query pipeline. It reuses the safe Apache AGE schema extraction and read-only NL-to-Cypher pattern from `nl2cypher_forpsql.ipynb`, pointed at `realdata_knowledge_spine`.


In [ ]:
# Install only if your notebook environment does not already have these packages.
# ! pip install openai "psycopg[binary]" python-dotenv


In [ ]:
import csv
import json
import os
import re

import psycopg
from dotenv import load_dotenv
from openai import AzureOpenAI

load_dotenv(override=True)

PG_HOST = os.getenv("PG_HOST")
PG_PORT = os.getenv("PG_PORT")
PG_DATABASE = os.getenv("PG_DATABASE")
PG_USER = os.getenv("PG_USER")
PG_PASSWORD = os.getenv("PG_PASSWORD")

GRAPH_NAME = "realdata_knowledge_spine"

client = AzureOpenAI(
    api_version="2024-12-01-preview",
    azure_endpoint="https://ciaiciath2-foundry-dev.cognitiveservices.azure.com/",
    api_key=os.getenv("AZURE_OPENAI_API_KEY"),
)

models = ["gpt-5.6-luna", "gpt-5.4-mini"]
NL2CYPHER_MODEL = models[0]
ANSWER_MODEL = models[0]


In [ ]:
def connect_postgres():
    return psycopg.connect(
        host=PG_HOST,
        port=PG_PORT,
        dbname=PG_DATABASE,
        user=PG_USER,
        password=PG_PASSWORD,
    )


def init_age(cursor):
    cursor.execute("LOAD 'age';")
    cursor.execute('SET search_path = ag_catalog, "$user", public;')


def open_age_connection():
    conn = connect_postgres()
    try:
        with conn.cursor() as cursor:
            init_age(cursor)
        conn.commit()
        return conn
    except Exception:
        conn.rollback()
        conn.close()
        raise


# Smoke test only. Query cells open their own short-lived connections.
test_conn = open_age_connection()
test_conn.close()
print("Connected to PostgreSQL AGE graph:", GRAPH_NAME)


In [ ]:
def normalize_agtype(value):
    if value is None:
        return None
    if isinstance(value, (list, dict, int, float, bool)):
        return value
    text = str(value).strip()
    try:
        return json.loads(text)
    except Exception:
        return text.strip('"')


In [ ]:
def get_age_node_schema(conn, graph_name):
    nodes = {}
    with conn.cursor() as cursor:
        init_age(cursor)
        cursor.execute(f"""
        SELECT *
        FROM cypher('{graph_name}', $$
            MATCH (n)
            RETURN DISTINCT labels(n), keys(n)
        $$) AS (labels agtype, properties agtype);
        """)
        rows = cursor.fetchall()

    for labels_value, properties_value in rows:
        labels = normalize_agtype(labels_value)
        properties = normalize_agtype(properties_value)
        if not isinstance(labels, list):
            labels = [labels]
        if not isinstance(properties, list):
            properties = [properties]
        for label in labels:
            nodes.setdefault(str(label), set()).update(str(prop) for prop in properties)
    return {label: sorted(properties) for label, properties in nodes.items()}


def get_age_relationship_schema(conn, graph_name):
    relationships = []
    with conn.cursor() as cursor:
        init_age(cursor)
        cursor.execute(f"""
        SELECT *
        FROM cypher('{graph_name}', $$
            MATCH (a)-[r]->(b)
            RETURN DISTINCT labels(a), type(r), labels(b)
        $$) AS (source_labels agtype, relationship agtype, target_labels agtype);
        """)
        rows = cursor.fetchall()

    for source_value, relationship_value, target_value in rows:
        relationships.append({
            "source": normalize_agtype(source_value),
            "relationship": normalize_agtype(relationship_value),
            "target": normalize_agtype(target_value),
        })
    return relationships


def get_age_graph_schema(conn, graph_name):
    return {
        "nodes": get_age_node_schema(conn, graph_name),
        "relationships": get_age_relationship_schema(conn, graph_name),
    }


In [ ]:
def build_schema_text(schema):
    lines = ["NODE LABELS AND PROPERTIES"]
    for label, properties in schema["nodes"].items():
        lines.append(f"\nNode: {label}")
        lines.append("Properties: " + ", ".join(properties))

    lines.append("\nRELATIONSHIPS")
    for relation in schema["relationships"]:
        source = relation["source"]
        target = relation["target"]
        if isinstance(source, list):
            source = ", ".join(source)
        if isinstance(target, list):
            target = ", ".join(target)
        lines.append(f"({source})-[:{relation['relationship']}]->({target})")
    return "\n".join(lines)


schema_conn = open_age_connection()
try:
    age_schema = get_age_graph_schema(schema_conn, GRAPH_NAME)
finally:
    schema_conn.close()

GRAPH_SCHEMA = build_schema_text(age_schema)
print(GRAPH_SCHEMA)


In [ ]:
FORBIDDEN_CYPHER = [
    "CREATE",
    "MERGE",
    "DELETE",
    "DETACH",
    "SET",
    "REMOVE",
    "DROP",
    "LOAD CSV",
    "FOREACH",
    "CALL",
]


def validate_read_only_cypher(cypher):
    normalized = cypher.upper().strip()
    for keyword in FORBIDDEN_CYPHER:
        if re.search(rf"\b{re.escape(keyword)}\b", normalized):
            raise ValueError(f"Unsafe Cypher detected: {keyword}")
    if not re.match(r"^(MATCH|OPTIONAL MATCH|WITH|UNWIND)\b", normalized):
        raise ValueError("Cypher must start with a read-only clause")
    return True


def validate_column_name(name):
    if not re.fullmatch(r"[A-Za-z_][A-Za-z0-9_]*", name):
        raise ValueError(f"Invalid column name: {name}")
    return name


In [ ]:
def llm_json(messages, model):
    response = client.chat.completions.create(
        model=model,
        messages=messages,
        response_format={"type": "json_object"},
    )
    return json.loads(response.choices[0].message.content)


def generate_age_query(question):
    messages = [
        {
            "role": "system",
            "content": """
You generate read-only Apache AGE Cypher queries.
Return JSON only with this exact shape:
{"cypher": "MATCH ... RETURN ...", "columns": ["column_1"]}

Rules:
- Use only the supplied schema.
- Never invent labels, properties, or relationships.
- Do not include the PostgreSQL SELECT FROM cypher wrapper.
- Every RETURN expression must have an explicit alias.
- Prefer scalar properties instead of full vertices or edges.
- Read-only Cypher only.
""".strip(),
        },
        {
            "role": "user",
            "content": f"GRAPH SCHEMA:\n{GRAPH_SCHEMA}\n\nQUESTION:\n{question}",
        },
    ]
    result = llm_json(messages, NL2CYPHER_MODEL)
    cypher = result["cypher"].strip()
    columns = [validate_column_name(column) for column in result["columns"]]
    validate_read_only_cypher(cypher)
    return {"cypher": cypher, "columns": columns}


In [ ]:
def execute_age_query(query_spec):
    cypher = query_spec["cypher"]
    columns = [validate_column_name(column) for column in query_spec["columns"]]
    validate_read_only_cypher(cypher)

    column_definition = ", ".join(f"{column} agtype" for column in columns)
    sql = f"""
    SELECT *
    FROM cypher('{GRAPH_NAME}', $$
        {cypher}
    $$) AS ({column_definition});
    """

    conn = open_age_connection()
    try:
        with conn.cursor() as cursor:
            cursor.execute(sql)
            rows = cursor.fetchall()
        conn.commit()
    except Exception:
        conn.rollback()
        raise
    finally:
        conn.close()

    return [
        {column: normalize_agtype(value) for column, value in zip(columns, row)}
        for row in rows
    ]


In [ ]:
def generate_answer(question, query_result):
    messages = [
        {
            "role": "system",
            "content": "Answer using only the supplied graph query result. If empty, say no matching data was found. Be concise.",
        },
        {
            "role": "user",
            "content": f"QUESTION:\n{question}\n\nQUERY RESULT:\n{json.dumps(query_result, indent=2, default=str)}",
        },
    ]
    response = client.chat.completions.create(
        model=ANSWER_MODEL,
        messages=messages,
    )
    return response.choices[0].message.content.strip()


def ask_age_graph(question, show_cypher=True, show_raw_result=False):
    query_spec = generate_age_query(question)
    query_result = execute_age_query(query_spec)
    answer = generate_answer(question, query_result)

    if show_cypher:
        print("Generated Cypher:\n")
        print(query_spec["cypher"])
    if show_raw_result:
        print("\nRaw AGE Result:\n")
        print(json.dumps(query_result, indent=2, default=str))
    return answer


In [ ]:
# Example questions after real_tranformation.ipynb has loaded the graph.
answer = ask_age_graph("List the available dataset entities and their domains", show_raw_result=True)
print(answer)


In [ ]:
answer = ask_age_graph("How can I setup MMM usecase?", show_raw_result=True)
print(answer)

In [ ]:
answer = ask_age_graph("How can I calculate ROI?", show_raw_result=True)
print(answer)

In [ ]:
answer = ask_age_graph("How Market share is calculated?", show_raw_result=True)
print(answer)

In [ ]:
answer = ask_age_graph("What is temperature in Hyderabad?", show_raw_result=True)
print(answer)

In [ ]:
answer = ask_age_graph("Define HCP 360 degree view analysis", show_raw_result=True)
print(answer)

In [ ]:
test_questions = [
    "What is HCP segmentation?",
    "What is Market Mix Modeling in the Commercial Pharma?",
    "How is RoI calculated in the Market Mix Modeling?",
    "What is specialty pharmacy?",
    "Give me information on the 340B sales program",
    "Can you tell me about LoT (Line of Therapy)?",
    "What is secondary market research?",
    "What is market access?",
    "What is the difference between qualitative and quantitative research?",
    "What is Gross-to-Net (GTN) in commercial pharma, and which deductions are usually included?",
    "How do formulary tiers, prior authorization, and step therapy affect market access?",
    "What is the difference between TRx, NRx, and NBRx in pharma analytics?",
    "What are patient support or hub services in specialty pharma, and how do they help with access and adherence"
]

In [ ]:
def ask_age_graph_record(question):
    record = {
        "question": question,
        "answer": "",
        "cypher query": "",
        "structured response": "",
    }

    query_spec = None
    try:
        query_spec = generate_age_query(question)
        query_result = execute_age_query(query_spec)
        answer = generate_answer(question, query_result)

        record["answer"] = answer
        record["cypher query"] = query_spec["cypher"]
        record["structured response"] = json.dumps(
            {
                "columns": query_spec["columns"],
                "rows": query_result,
            },
            default=str,
            ensure_ascii=False,
        )
    except Exception as exc:
        record["answer"] = f"ERROR: {exc}"
        if query_spec:
            record["cypher query"] = query_spec.get("cypher", "")
        record["structured response"] = json.dumps(
            {
                "error": str(exc),
                "query_spec": query_spec,
            },
            default=str,
            ensure_ascii=False,
        )

    return record


def export_question_answers_to_csv(questions, csv_path="real_graph_qa_results.csv"):
    fieldnames = [
        "question",
        "answer",
        "cypher query",
        "structured response",
    ]

    records = []
    for index, question in enumerate(questions, start=1):
        print(f"Running {index}/{len(questions)}: {question}")
        records.append(ask_age_graph_record(question))

    with open(csv_path, "w", newline="", encoding="utf-8") as handle:
        writer = csv.DictWriter(handle, fieldnames=fieldnames)
        writer.writeheader()
        writer.writerows(records)

    print("Saved CSV:", csv_path)
    return records


# Run this cell after `test_questions` is defined.
qa_records = export_question_answers_to_csv(
    test_questions,
    csv_path="real_graph_qa_results.csv",
)
qa_records[:2]
